# 04 — Train: persistence baseline

Evaluates a persistence baseline over the configured direct-forecast horizon for the target station, on exactly the cohort and fold windows the fitted candidates use.

**Inputs:** joined train/test feature artifacts and their metadata contract
**Outputs:** in-notebook prediction preview/test metrics and an MLflow run hierarchy

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins the notebook's constants. There is no model configuration to pin: persistence has no hyperparameters and nothing to fit. The joined feature metadata is still the source of truth for the column contract — the target station's engineered features plus every retained station's raw measurements at issue time `t` — because the baseline has to qualify rows the same way the fitted candidates do.

**Parameters**

The values below are illustrative examples, not run configuration. Imported values from `src/config.py` and executable constants in this notebook are authoritative for a run; Stage-3 feature metadata is authoritative for the realized predictor and target columns.

| Parameter | Example value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `PERSISTENCE_COLUMN` | `{TARGET_STATION_ID}__water_level` | The single column the forecast is built from: the target station's observed water level at issue time `t`. Checked against the metadata contract at load time, so a Stage-3 rename fails loudly instead of surfacing as a `KeyError` mid-fold. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `_H` | The metadata-declared future water levels, one per lead hour. The baseline emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |
| `FORECAST_HORIZON_HOURS` | `24` | Example configured horizon; execution uses the imported value and validates it against the metadata contract width. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds, matching the fitted candidates. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `MLFLOW_EXPERIMENT_NAME` | `"persistence"` | Experiment receiving the parent, nested fold, and sealed-test runs. |

In [ ]:
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import (
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    regime_aggregate_figure,
    regime_horizon_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import (
    regime_mlflow_metrics,
    regime_mlflow_params,
    sealed_test_regime_tables,
)
from src.training import (
    prediction_preview,
    summarize_cv_metrics,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
MLFLOW_EXPERIMENT_NAME = "persistence"
PREDICTION_PREVIEW_ROWS = 5
station_id = TARGET_STATION_ID
PERSISTENCE_COLUMN = f"{station_id}__water_level"

## Shared evaluation cohort

Every stage-4 candidate is fit and scored on exactly the same rows, which is what makes their reported numbers comparable to each other and to this baseline. One row is one *issue time* `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true and all $H$ future water levels `t+1 … t+H` were actually observed. This drops issue times sitting near a data gap, plus the final $H$ hours of each artifact, which have no complete future.
2. **Every full-contract predictor is present.** All metadata-declared predictors must be available: the target station's engineered features — whose lag and rolling terms require their complete configured lookbacks and therefore create an initial warm-up — plus every retained station's raw water level, imputation flag, precipitation, and temperature at `t`.

Eligibility deliberately uses the **full** predictor contract even though persistence reads a single column. A baseline scored on a different, larger cohort than the models it benchmarks is not a baseline: the extra rows it would gain are precisely the ones the fitted candidates could not use, and any difference in the reported numbers would then be part cohort, part skill. The cost is that persistence's absolute error here is slightly worse than a standalone persistence forecast run over every hour with an observed water level — a deliberate trade for a like-for-like comparison.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic — not even a scaler mean — is ever computed from it.

## Shared helpers

The joined dataset — contract loading, common-cohort preparation, and chronological folds — comes from `src.dataset`, so this baseline cannot drift from the cohort the fitted candidates use. Prediction checks, metric tables, CV summaries, and previews come from `src.training` and `src.metrics`, and the evaluation figures from `src.plots`. The only notebook-local code is the forecast itself, which is one `np.repeat`.

## Load the joined dataset

`load_joined_dataset()` does the whole preamble in one call — the same call the fitted candidates make. It reads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory and checks their station, horizon, and column contracts before any scoring; `PERSISTENCE_COLUMN` is then checked against the declared predictors so a Stage-3 column rename fails here rather than deep inside the fold loop.

It also applies the eligibility cohort to each artifact independently and builds the validation folds, so an empty cohort stops the notebook rather than yielding a metric computed from nothing. Persistence has no fitted parameters, so the train cohort exists purely to define the folds below. Not a single train *value* reaches any forecast.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
if PERSISTENCE_COLUMN not in FULL_FEATURE_COLUMNS:
    raise ValueError(f"{PERSISTENCE_COLUMN!r} is not in the joined predictor contract")
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## The forecast

The whole model is this: *whatever the water level is now, that is the forecast for every configured lead hour.* `np.repeat` takes the single observed water-level column at each issue time and tiles it across `len(TARGET_COLUMNS)` columns, producing the same `(n_issue_times, H)` shape every fitted candidate produces.

- `repeats=len(TARGET_COLUMNS)` — one copy per configured lead hour.
- `axis=1` — repeat along columns (horizons), not rows. `axis=0` would instead duplicate issue times and silently break the alignment with the scored frame.

In [ ]:
def persistence_predictions(rows: pd.DataFrame) -> np.ndarray:
    """Tile each issue time's observed water level across every forecast horizon."""
    return np.repeat(
        rows[[PERSISTENCE_COLUMN]].to_numpy(dtype=float),
        repeats=len(TARGET_COLUMNS),
        axis=1,
    )

## Cross-validation folds

Nothing is fitted here — the folds exist so this baseline reports per-fold numbers on the *same* validation windows the fitted candidates report, which is what makes a fold-by-fold comparison in Stage 5/6 meaningful. Eligible training rows are sorted by issue time, `TimeSeriesSplit` creates `N_VALIDATION_FOLDS` expanding windows with the same explicit `test_size` and the same `EMBARGO_HOURS`-row gap, and only each fold's validation slice is scored.

Comparability holds by construction: this notebook and the fitted ones import the same `src.config` constants and use the same full-contract eligibility, so an equal eligible-row count implies identical fold indices. `common_train_rows` and both input Parquet SHA-256s are logged so a mismatch is detectable downstream rather than silent.

One MLflow parent run receives the CV summary; each fold gets a nested child run.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splits = dataset.folds
validation_test_size = dataset.validation_test_size
fold_aggregate_rows = []
fold_horizon_rows = []
with mlflow.start_run(
    run_name="persistence_cv",
    nested=False,
    tags={
        "phase": "cv",
        "run_type": "candidate_parent",
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "cv",
            "run_type": "candidate_parent",
            "persistence_column": PERSISTENCE_COLUMN,
            **INPUT_PARQUET_SHA256_PARAMS,
            "n_validation_folds": N_VALIDATION_FOLDS,
            "validation_test_size": validation_test_size,
            "embargo_hours": EMBARGO_HOURS,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "common_train_rows": len(train_rows),
        }
    )

    # Each fold's training indices are intentionally unused: persistence never fits.
    for fold_number, (_fold_train_indices, fold_validation_indices) in enumerate(
        cv_splits, start=1
    ):
        with mlflow.start_run(
            run_name=f"persistence_cv_fold_{fold_number}",
            nested=True,
            tags={
                "phase": "cv",
                "run_type": "fold",
                "fold": str(fold_number),
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            fold_validation_rows = train_rows.iloc[fold_validation_indices]
            fold_predictions = validate_predictions(
                persistence_predictions(fold_validation_rows),
                expected_rows=len(fold_validation_rows),
                target_columns=TARGET_COLUMNS,
                artifact_name="fold",
            )
            fold_aggregate, fold_per_horizon = metric_tables(
                fold_validation_rows[TARGET_COLUMNS],
                fold_predictions,
                target_columns=TARGET_COLUMNS,
                station_id=station_id,
            )
            fold_aggregate_rows.append(fold_aggregate.iloc[0])
            fold_horizon_rows.append(fold_per_horizon)
            mlflow.log_params(
                {
                    "phase": "cv",
                    "run_type": "fold",
                    "persistence_column": PERSISTENCE_COLUMN,
                    **INPUT_PARQUET_SHA256_PARAMS,
                    "fold": fold_number,
                    "validation_rows": len(fold_validation_rows),
                    "gap_rows": EMBARGO_HOURS,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "validation_start": fold_validation_rows["timestamp"]
                    .iloc[0]
                    .isoformat(),
                    "validation_end": fold_validation_rows["timestamp"]
                    .iloc[-1]
                    .isoformat(),
                    "validation_index_start": int(fold_validation_indices[0]),
                    "validation_index_end": int(fold_validation_indices[-1]),
                }
            )
            mlflow.log_metrics(
                {
                    "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                    "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                    "fold_me": float(fold_aggregate.iloc[0]["me"]),
                    "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                    **{
                        f"fold_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                        for row in fold_per_horizon.itertuples()
                    },
                    **{
                        f"fold_me_horizon_{row.horizon_hours:02d}": float(row.me)
                        for row in fold_per_horizon.itertuples()
                    },
                    **{
                        f"fold_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                        for row in fold_per_horizon.itertuples()
                    },
                    **{
                        f"fold_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                        for row in fold_per_horizon.itertuples()
                    },
                }
            )

    fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
    fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_metrics = summarize_cv_metrics(fold_aggregate_metrics, fold_horizon_metrics)
    mlflow.log_metrics(cv_metrics)
print(f"Persistence cross-validation results for {station_id}")
display(fold_aggregate_metrics.reset_index(drop=True))

## Evaluate on the sealed test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE/ME/R², the same metrics for each configured lead in the direct forecast, and a short preview against the actual targets. There is no second pass.

This is the reference every other stage-4 notebook has to beat. River level is smooth and strongly autocorrelated, which makes persistence genuinely hard to improve on at `t+1` and progressively easier toward the final configured horizon — so the per-horizon table matters more than the aggregate one here. A candidate that cannot beat these numbers has not learned anything worth keeping.

In [ ]:
test_predictions = validate_predictions(
    persistence_predictions(test_rows),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
regime_definition, regime_aggregate_metrics, regime_horizon_metrics = (
    sealed_test_regime_tables(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        target_columns=TARGET_COLUMNS,
        station_id=station_id,
        quartile_cutoffs_cm=dataset.target_water_level_quartile_cutoffs_cm,
        quartile_reference_count=dataset.target_water_level_quartile_reference_count,
    )
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Persistence reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Persistence reported non-finite horizon metrics")
with mlflow.start_run(
    run_name="persistence_test",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "persistence_column": PERSISTENCE_COLUMN,
            **INPUT_PARQUET_SHA256_PARAMS,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "common_train_rows": len(train_rows),
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_rmse_mae_boxplots_fig = cv_error_boxplots_figure(
        fold_horizon_metrics,
        TARGET_COLUMNS,
        title="Persistence CV errors",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    test_error_boxplots_fig = test_error_boxplots_figure(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title="Persistence final-test errors",
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title="Persistence predicted vs actual",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
print(f"Persistence test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

## Q1–Q4 aggregate diagnostics


In [ ]:
display(
    regime_aggregate_figure(
        regime_aggregate_metrics, regime_group="quartile", model_label="Persistence"
    )
)

## Q1–Q4 MAE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="mae",
        model_label="Persistence",
    )
)

## Q1–Q4 RMSE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="rmse",
        model_label="Persistence",
    )
)

## Q1–Q4 ME by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="quartile",
        metric="me",
        model_label="Persistence",
    )
)

## Alarm aggregate diagnostics


In [ ]:
display(
    regime_aggregate_figure(
        regime_aggregate_metrics, regime_group="alarm", model_label="Persistence"
    )
)

## Alarm MAE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="mae",
        model_label="Persistence",
    )
)

## Alarm RMSE by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="rmse",
        model_label="Persistence",
    )
)

## Alarm ME by forecast horizon


In [ ]:
display(
    regime_horizon_figure(
        regime_horizon_metrics,
        regime_group="alarm",
        metric="me",
        model_label="Persistence",
    )
)